In [1]:
import os, json, shutil, math
from pathlib import Path

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from tqdm import tqdm
from glob import glob
import pandas as pd

import contextlib
from scipy.ndimage import label as cc_label

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast


SPLITS_2020 = Path(r"D:\master_experiments\data\splits\BraTS2020_Splits")
SPLITS_2024 = Path(r"D:\master_experiments\data\splits\BraTS2024_Splits")

MOME_2020_BEST = Path(r"D:\master_experiments\models_configs\MoME_BraTS2020\checkpoints\mome_best.pth")
MOME_2024_BEST = Path(r"D:\master_experiments\models_configs\MoME_BraTS2024\checkpoints\mome_best.pth")

CROSS_BASE        = Path(r"D:\master_experiments\models_configs\cross_eval\MoME")
PRED_2020_ON_2024 = CROSS_BASE / "preds_2020_on_2024"
PRED_2024_ON_2020 = CROSS_BASE / "preds_2024_on_2020"
LOGS              = CROSS_BASE / "logs"
for p in [PRED_2020_ON_2024, PRED_2024_ON_2020, LOGS]:
    p.mkdir(parents=True, exist_ok=True)

N_EXPERTS = 4
DEPTH     = 3
BASE_CH   = 32
PATCH     = 128

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

MODS_2020 = ["flair", "t1",  "t1ce", "t2"]
MODS_2024 = ["t2f",   "t1n", "t1c",  "t2w"]

print("Device:", DEVICE)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory/1e9:.1f} GB")
print(f"  Pesos MoME 2020 existem: {MOME_2020_BEST.exists()}")
print(f"  Pesos MoME 2024 existem: {MOME_2024_BEST.exists()}")

Device: cuda
GPU: NVIDIA GeForce RTX 4070 SUPER | VRAM: 12.9 GB
  Pesos MoME 2020 existem: True
  Pesos MoME 2024 existem: True


In [2]:
with open(SPLITS_2020 / "splits_metadata.json", "r", encoding="utf-8") as f:
    meta_2020 = json.load(f)
with open(SPLITS_2024 / "splits_metadata.json", "r", encoding="utf-8") as f:
    meta_2024 = json.load(f)

test_ids_2020 = meta_2020["ids"]["test"]
test_ids_2024 = meta_2024["ids"]["test"]

print(f"Test BraTS2020: {len(test_ids_2020)} casos")
print(f"Test BraTS2024: {len(test_ids_2024)} casos")

Test BraTS2020: 53 casos
Test BraTS2024: 203 casos


In [3]:
def case_dir_2020(cid): return SPLITS_2020 / "test" / cid
def case_dir_2024(cid): return SPLITS_2024 / "test" / cid

def find_file_2020(folder: Path, key: str) -> Path:
    for cand in [folder / f"{key}.nii.gz", folder / f"{key}.nii"]:
        if cand.exists(): return cand
    cands = sorted(folder.glob(f"*{key}*.nii*"))
    if key == "t1":
        cands = [c for c in cands if "t1ce" not in c.name.lower()]
    if not cands:
        raise FileNotFoundError(f"{key} not found in {folder}")
    return cands[0]

def find_file_2024(folder: Path, key: str) -> Path:
    for cand in [folder / f"{key}.nii.gz", folder / f"{key}.nii"]:
        if cand.exists(): return cand
    cands = sorted(folder.glob(f"*{key}*.nii*"))
    if key == "t1n":
        cands = [c for c in cands if "t1c" not in c.name.lower()]
    if not cands:
        raise FileNotFoundError(f"{key} not found in {folder}")
    return cands[0]

def load_arr(p):
    return np.asanyarray(nib.load(str(p)).dataobj)

def norm_zscore_fg(x): 
    x = x.astype(np.float32)
    mask = x > 0
    if mask.sum() == 0:
        return np.zeros_like(x, dtype=np.float32)
    mean = float(x[mask].mean())
    std  = float(x[mask].std()) + 1e-8
    out  = (x - mean) / std
    out[~mask] = 0.0
    return out

def dice_score(a, b):
    inter = np.count_nonzero(a & b)
    denom = np.count_nonzero(a) + np.count_nonzero(b)
    return 1.0 if denom == 0 else (2.0 * inter / denom)

def load_seg_2020(cid):
    p = find_file_2020(case_dir_2020(cid), "seg")
    data = load_arr(p).astype(np.int16)
    data[data == 4] = 3
    return data

def load_seg_2024(cid):
    p = find_file_2024(case_dir_2024(cid), "seg")
    return load_arr(p).astype(np.int16)

print("Helpers OK")

Helpers OK


In [4]:
def double_conv(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
        nn.InstanceNorm3d(out_ch),
        nn.LeakyReLU(0.01, inplace=True),
        nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
        nn.InstanceNorm3d(out_ch),
        nn.LeakyReLU(0.01, inplace=True),
    )


class ExpertUNet(nn.Module):
    def __init__(self, n_cls, base_ch=BASE_CH, depth=DEPTH):
        super().__init__()
        self.depth = depth
        chs = [base_ch * (2 ** i) for i in range(depth + 1)]

        self.enc  = nn.ModuleList()
        self.pool = nn.ModuleList()
        ch_in = 1
        for i in range(depth):
            self.enc.append(double_conv(ch_in, chs[i]))
            self.pool.append(nn.MaxPool3d(2))
            ch_in = chs[i]

        self.bn = double_conv(ch_in, chs[depth])

        self.up   = nn.ModuleList()
        self.dec  = nn.ModuleList()
        self.head = nn.ModuleList()
        for i in range(depth - 1, -1, -1):
            self.up.append(nn.ConvTranspose3d(chs[i+1], chs[i], 2, stride=2))
            self.dec.append(double_conv(chs[i] * 2, chs[i]))
            self.head.append(nn.Conv3d(chs[i], n_cls, 1))

        self.feat_chs = [base_ch * (2 ** i) for i in range(depth)]

    def forward(self, x):
        skips = []
        for enc, pool in zip(self.enc, self.pool):
            x = enc(x)
            skips.append(x)
            x = pool(x)

        x = self.bn(x)

        logits_list, feat_list = [], []
        for i, (up, dec, head) in enumerate(
                zip(self.up, self.dec, self.head)):
            x = up(x)
            x = dec(torch.cat([skips[-(i+1)], x], dim=1))
            feat_list.append(x)
            logits_list.append(head(x))

        return logits_list[::-1], feat_list[::-1]


class GatingNetwork(nn.Module):
    def __init__(self, n_experts, feat_chs_per_level, gate_ch=16):
        super().__init__()
        self.n_experts = n_experts
        self.heads = nn.ModuleList()
        for ch in feat_chs_per_level:
            in_ch = ch * n_experts
            self.heads.append(nn.Sequential(
                nn.Conv3d(in_ch, gate_ch, 3, padding=1, bias=False),
                nn.InstanceNorm3d(gate_ch),
                nn.LeakyReLU(0.01, inplace=True),
                nn.Conv3d(gate_ch, n_experts, 1),
            ))

    def forward(self, feat_lists):
        n_levels  = len(feat_lists[0])
        gate_maps = []
        for lvl in range(n_levels):
            cat   = torch.cat([feat_lists[i][lvl]
                               for i in range(self.n_experts)], dim=1)
            raw   = self.heads[lvl](cat)
            gates = torch.softmax(raw, dim=1)
            gate_maps.append(gates)
        return gate_maps


class MoME(nn.Module):
    def __init__(self, n_experts=N_EXPERTS, n_cls=4,
                 base_ch=BASE_CH, depth=DEPTH):
        super().__init__()
        self.n_experts = n_experts
        self.depth     = depth
        self.n_levels  = depth

        self.experts = nn.ModuleList([
            ExpertUNet(n_cls, base_ch, depth)
            for _ in range(n_experts)
        ])

        feat_chs = self.experts[0].feat_chs
        self.gating = GatingNetwork(n_experts, feat_chs, gate_ch=16)

    def forward(self, x, mode="mome", i_expert=None):
        if mode == "expert":
            assert i_expert is not None
            assert x.shape[1] == 1
            logits, _ = self.experts[i_expert](x)
            return logits

        all_logits = []
        all_feats  = []
        for i in range(self.n_experts):
            xi = x[:, i:i+1]
            logits, feats = self.experts[i](xi)
            all_logits.append(logits)
            all_feats.append(feats)

        gate_maps = self.gating(all_feats)

        ensemble_logits = []
        for lvl in range(self.n_levels):
            o = sum(
                all_logits[i][lvl] * gate_maps[lvl][:, i:i+1]
                for i in range(self.n_experts)
            )
            ensemble_logits.append(o)

        return ensemble_logits, all_logits

print("Arquitetura MoME OK")

Arquitetura MoME OK


In [5]:
def _gaussian_kernel_3d(size: int) -> np.ndarray:
    sigma = size / 8
    ax    = np.arange(size) - size // 2
    g     = np.exp(-0.5 * (ax / sigma) ** 2)
    g3d   = g[:, None, None] * g[None, :, None] * g[None, None, :]
    return (g3d / g3d.max()).astype(np.float32)


def sliding_window_inference(model: MoME, vol_4ch: np.ndarray,
                              n_classes: int,
                              patch=PATCH, overlap=0.5) -> np.ndarray:
    model.eval()
    _, D, H, W = vol_4ch.shape
    step       = max(int(patch * (1.0 - overlap)), 1)

    accum   = np.zeros((n_classes, D, H, W), dtype=np.float32)
    weights = np.zeros((D, H, W), dtype=np.float32)
    gauss   = _gaussian_kernel_3d(patch)

    def _make_range(dim):
        last = max(dim - patch, 0)
        r    = list(range(0, last + 1, step))
        if r and r[-1] != last:
            r.append(last)
        return r if r else [0]

    with torch.no_grad():
        for z0 in _make_range(D):
            for y0 in _make_range(H):
                for x0 in _make_range(W):
                    z1 = min(z0 + patch, D); z0_ = z1 - patch
                    y1 = min(y0 + patch, H); y0_ = y1 - patch
                    x1 = min(x0 + patch, W); x0_ = x1 - patch

                    patch_np = vol_4ch[:, z0_:z1, y0_:y1, x0_:x1]
                    t = torch.from_numpy(patch_np[None]).to(DEVICE)

                    with autocast("cuda") if USE_AMP else contextlib.nullcontext():
                        ens_logits, _ = model(t, mode="mome")
                        prob = torch.softmax(ens_logits[0], dim=1)
                        prob = prob[0].cpu().float().numpy()

                    accum[:, z0_:z1, y0_:y1, x0_:x1] += prob * gauss[None]
                    weights[z0_:z1, y0_:y1, x0_:x1]  += gauss

    accum /= np.maximum(weights[None], 1e-8)
    return accum.argmax(axis=0).astype(np.int16)

print("sliding_window_inference OK")

sliding_window_inference OK


In [6]:
LW_MIN_SIZE = 50 

def lesion_wise_dice(gt_mask: np.ndarray, pr_mask: np.ndarray, min_size: int = LW_MIN_SIZE) -> float:
    structure = np.ones((3, 3, 3), dtype=int)
    if not gt_mask.any() and not pr_mask.any():
        return 1.0
    gt_lab, n_gt = cc_label(gt_mask.astype(bool), structure=structure)
    pr_lab, n_pr = cc_label(pr_mask.astype(bool), structure=structure)
    dice_list = []
    matched_pred = set()
    for g in range(1, n_gt + 1):
        gt_l = (gt_lab == g)
        if gt_l.sum() < min_size:
            continue
        overlap_ids = np.unique(pr_lab[gt_l])
        overlap_ids = overlap_ids[overlap_ids > 0]
        if len(overlap_ids) == 0:
            dice_list.append(0.0)
        else:
            pr_match = np.isin(pr_lab, overlap_ids)
            inter = np.logical_and(gt_l, pr_match).sum()
            denom = gt_l.sum() + pr_match.sum()
            dice_list.append(2.0 * inter / denom if denom > 0 else 0.0)
            matched_pred.update(int(p) for p in overlap_ids)
    for p in range(1, n_pr + 1):
        if p in matched_pred:
            continue
        pr_l = (pr_lab == p)
        if pr_l.sum() < min_size:
            continue
        dice_list.append(0.0)
    return float(np.mean(dice_list)) if dice_list else 1.0

print("lesion_wise_dice OK")

lesion_wise_dice OK


In [7]:
# Direcao A: MoME treinado em BraTS2020 -> predito sobre teste BraTS2024

model_A = MoME(n_experts=N_EXPERTS, n_cls=4, base_ch=BASE_CH, depth=DEPTH).to(DEVICE)
model_A.load_state_dict(torch.load(MOME_2020_BEST, map_location=DEVICE))
model_A.eval()
print(f"MoME BraTS2020 carregado (n_cls=4): {sum(p.numel() for p in model_A.parameters())/1e6:.1f}M params")

# Inferencia sobre fracao de teste do BraTS2024
for cid in tqdm(test_ids_2024, desc="Predicao (A: 2020->2024)"):
    out_path = PRED_2020_ON_2024 / f"{cid}.nii.gz"
    d   = case_dir_2024(cid)
    ref = nib.load(str(find_file_2024(d, "t2f"))) 
    imgs = [norm_zscore_fg(load_arr(find_file_2024(d, m))) for m in MODS_2024]
    vol  = np.stack(imgs, axis=0).astype(np.float32) 

    pred = sliding_window_inference(model_A, vol, n_classes=4) 
    nib.save(nib.Nifti1Image(pred, ref.affine, ref.header), str(out_path))

del model_A
torch.cuda.empty_cache()
print("Predicoes direcao A concluidas.")

C:\Users\dados\AppData\Local\Temp\ipykernel_24088\3862872131.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_A.load_state_dict(torch.load(MOME_2020_BEST, map_locat

MoME BraTS2020 carregado (n_cls=4): 22.8M params


Predicao (A: 2020->2024): 100%|██████████| 203/203 [13:04<00:00,  3.87s/it]

Predicoes direcao A concluidas.


In [8]:
# Metrica 1 (A): Dice GLOBAL - modelo 2020 sobre teste 2024

rows = []
for cid in tqdm(test_ids_2024, desc="Dice global por caso (A: 2020->2024)"):
    gt = load_seg_2024(cid)  # nativo {0,1,2,3,4}
    pr = load_arr(PRED_2020_ON_2024 / f"{cid}.nii.gz").astype(np.int16) 

    wt = dice_score(((gt==1)|(gt==2)|(gt==3)), ((pr==1)|(pr==2)|(pr==3)))
    tc = dice_score(((gt==1)|(gt==3)),         ((pr==1)|(pr==3)))
    et = dice_score((gt==3),                   (pr==3))

    rows.append({"id": cid, "dice_WT": wt, "dice_TC": tc, "dice_ET": et})

df_A = pd.DataFrame(rows)
df_A.to_csv(LOGS / "cross_A_2020_on_2024_dice.csv", index=False)
df_A[["dice_WT","dice_TC","dice_ET"]].describe().round(4)

Dice global por caso (A: 2020->2024): 100%|██████████| 203/203 [00:30<00:00,  6.65it/s]


,dice_WT,dice_TC,dice_ET
count,203.0000,203.0000,203.0000
mean,0.7411,0.3787,0.4922
std,0.2157,0.3794,0.3803
min,0.0000,0.0000,0.0000
25%,0.6769,0.0000,0.0000
50%,0.8022,0.2893,0.6040
75%,0.8895,0.7437,0.8425
max,0.9796,1.0000,1.0000


In [9]:
# Analise condicional 1/3 (A): Frequencia de presenca das regioes compostas no GT 2024

presence_rows = []
for cid in tqdm(test_ids_2024, desc="Presenca GT por caso (A)"):
    gt = load_seg_2024(cid)
    presence_rows.append({
        "id":     cid,
        "has_WT": bool(((gt==1)|(gt==2)|(gt==3)).any()),
        "has_TC": bool(((gt==1)|(gt==3)).any()),
        "has_ET": bool((gt==3).any()),
    })
pres_df_A = pd.DataFrame(presence_rows)
df_full_A = df_A.merge(pres_df_A, on="id")

df_presence_A = pd.DataFrame([
    {"region":     col,
     "n_presente": int(df_full_A[col].sum()),
     "n_total":    len(df_full_A),
     "pct":        100.0 * df_full_A[col].sum() / len(df_full_A)}
    for col in ["has_WT", "has_TC", "has_ET"]
])
df_presence_A.round(2)

Presenca GT por caso (A): 100%|██████████| 203/203 [00:16<00:00, 12.39it/s]


,region,n_presente,n_total,pct
0,has_WT,203,203,100.00
1,has_TC,152,203,74.88
2,has_ET,150,203,73.89


In [10]:
# Analise condicional 2/3 (A): Dice CONDICIONAL

pairs = [("dice_WT","has_WT"), ("dice_TC","has_TC"), ("dice_ET","has_ET")]
df_conditional_A = pd.DataFrame({
    col: df_full_A.loc[df_full_A[has], col].reset_index(drop=True)
    for col, has in pairs
})
df_conditional_A.to_csv(LOGS / "cross_A_2020_on_2024_conditional.csv", index=False)
df_conditional_A.describe().round(4)

,dice_WT,dice_TC,dice_ET
count,203.0000,152.0000,150.0000
mean,0.7411,0.4465,0.5595
std,0.2157,0.3536,0.3220
min,0.0000,0.0000,0.0000
25%,0.6769,0.0346,0.2962
50%,0.8022,0.4884,0.6630
75%,0.8895,0.7688,0.8361
max,0.9796,0.9815,0.9668


In [11]:
# Analise condicional 3/3 (A): Bimodalidade

df_bimodality_A = pd.DataFrame([
    {"region":         col,
     "pct_dice_zero": float((df_full_A[col] == 0.0).mean()) * 100,
     "pct_dice_one":  float((df_full_A[col] >= 0.999).mean()) * 100}
    for col, _ in pairs
])
df_bimodality_A.to_csv(LOGS / "cross_A_2020_on_2024_bimodality.csv", index=False)
df_bimodality_A.round(2)

,region,pct_dice_zero,pct_dice_one
0,dice_WT,1.97,0.00
1,dice_TC,29.06,4.43
2,dice_ET,26.11,7.88


In [12]:
# Metrica 3 (A): Lesion-wise Dice

lw_rows = []
for cid in tqdm(test_ids_2024, desc="Lesion-wise Dice por caso (A)"):
    gt = load_seg_2024(cid)
    pr = load_arr(PRED_2020_ON_2024 / f"{cid}.nii.gz").astype(np.int16)

    gt_wt = (gt==1)|(gt==2)|(gt==3)
    pr_wt = (pr==1)|(pr==2)|(pr==3)
    gt_tc = (gt==1)|(gt==3)
    pr_tc = (pr==1)|(pr==3)
    gt_et = (gt==3)
    pr_et = (pr==3)

    lw_rows.append({
        "id":     cid,
        "lwd_WT": lesion_wise_dice(gt_wt, pr_wt),
        "lwd_TC": lesion_wise_dice(gt_tc, pr_tc),
        "lwd_ET": lesion_wise_dice(gt_et, pr_et),
    })

lw_df_A = pd.DataFrame(lw_rows)
lw_df_A.to_csv(LOGS / "cross_A_2020_on_2024_lwd.csv", index=False)
lw_df_A[["lwd_WT","lwd_TC","lwd_ET"]].describe().round(4)

Lesion-wise Dice por caso (A): 100%|██████████| 203/203 [02:37<00:00,  1.29it/s]


,lwd_WT,lwd_TC,lwd_ET
count,203.0000,203.0000,203.0000
mean,0.5074,0.3144,0.4996
std,0.2907,0.3561,0.3758
min,0.0000,0.0000,0.0000
25%,0.2669,0.0000,0.1396
50%,0.4302,0.1666,0.4363
75%,0.8274,0.6260,0.8944
max,0.9796,1.0000,1.0000


In [13]:
# Direcao B: MoME treinado em BraTS2024 -> predito sobre teste BraTS2020 

model_B = MoME(n_experts=N_EXPERTS, n_cls=5, base_ch=BASE_CH, depth=DEPTH).to(DEVICE)
model_B.load_state_dict(torch.load(MOME_2024_BEST, map_location=DEVICE))
model_B.eval()
print(f"MoME BraTS2024 carregado (n_cls=5): {sum(p.numel() for p in model_B.parameters())/1e6:.1f}M params")

for cid in tqdm(test_ids_2020, desc="Predicao (B: 2024->2020)"):
    out_path = PRED_2024_ON_2020 / f"{cid}.nii.gz"
    d   = case_dir_2020(cid)
    ref = nib.load(str(find_file_2020(d, "flair")))
    imgs = [norm_zscore_fg(load_arr(find_file_2020(d, m))) for m in MODS_2020]
    vol  = np.stack(imgs, axis=0).astype(np.float32)  

    pred = sliding_window_inference(model_B, vol, n_classes=5)  
    pred[pred == 4] = 0  
    nib.save(nib.Nifti1Image(pred, ref.affine, ref.header), str(out_path))

del model_B
torch.cuda.empty_cache()
print("Predicoes direcao B concluidas.")

C:\Users\dados\AppData\Local\Temp\ipykernel_24088\2650734633.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_B.load_state_dict(torch.load(MOME_2024_BEST, map_locat

MoME BraTS2024 carregado (n_cls=5): 22.8M params


Predicao (B: 2024->2020): 100%|██████████| 53/53 [04:19<00:00,  4.89s/it]

Predicoes direcao B concluidas.


In [14]:
# Metrica 1 (B): Dice GLOBAL - modelo 2024 sobre teste 2020

rows = []
for cid in tqdm(test_ids_2020, desc="Dice global por caso (B: 2024->2020)"):
    gt = load_seg_2020(cid)  # remap 4->3
    pr = load_arr(PRED_2024_ON_2020 / f"{cid}.nii.gz").astype(np.int16) 

    wt = dice_score(((gt==1)|(gt==2)|(gt==3)), ((pr==1)|(pr==2)|(pr==3)))
    tc = dice_score(((gt==1)|(gt==3)),         ((pr==1)|(pr==3)))
    et = dice_score((gt==3),                   (pr==3))

    rows.append({"id": cid, "dice_WT": wt, "dice_TC": tc, "dice_ET": et})

df_B = pd.DataFrame(rows)
df_B.to_csv(LOGS / "cross_B_2024_on_2020_dice.csv", index=False)
df_B[["dice_WT","dice_TC","dice_ET"]].describe().round(4)

Dice global por caso (B: 2024->2020): 100%|██████████| 53/53 [00:04<00:00, 11.35it/s]


,dice_WT,dice_TC,dice_ET
count,53.0000,53.0000,53.0000
mean,0.8320,0.7168,0.7706
std,0.1019,0.2913,0.2019
min,0.5702,0.0026,0.0158
25%,0.7571,0.6467,0.7471
50%,0.8787,0.8646,0.8512
75%,0.9117,0.9009,0.8850
max,0.9524,0.9557,0.9329


In [15]:
# Analise condicional 1/3 (B): Frequencia de presenca no GT 2020

presence_rows = []
for cid in tqdm(test_ids_2020, desc="Presenca GT por caso (B)"):
    gt = load_seg_2020(cid)
    presence_rows.append({
        "id":     cid,
        "has_WT": bool(((gt==1)|(gt==2)|(gt==3)).any()),
        "has_TC": bool(((gt==1)|(gt==3)).any()),
        "has_ET": bool((gt==3).any()),
    })
pres_df_B = pd.DataFrame(presence_rows)
df_full_B = df_B.merge(pres_df_B, on="id")

df_presence_B = pd.DataFrame([
    {"region":     col,
     "n_presente": int(df_full_B[col].sum()),
     "n_total":    len(df_full_B),
     "pct":        100.0 * df_full_B[col].sum() / len(df_full_B)}
    for col in ["has_WT", "has_TC", "has_ET"]
])
df_presence_B.round(2)

Presenca GT por caso (B): 100%|██████████| 53/53 [00:02<00:00, 25.41it/s]


,region,n_presente,n_total,pct
0,has_WT,53,53,100.0
1,has_TC,53,53,100.0
2,has_ET,53,53,100.0


In [16]:
# Analise condicional 2/3 (B): Dice CONDICIONAL

pairs = [("dice_WT","has_WT"), ("dice_TC","has_TC"), ("dice_ET","has_ET")]
df_conditional_B = pd.DataFrame({
    col: df_full_B.loc[df_full_B[has], col].reset_index(drop=True)
    for col, has in pairs
})
df_conditional_B.to_csv(LOGS / "cross_B_2024_on_2020_conditional.csv", index=False)
df_conditional_B.describe().round(4)

,dice_WT,dice_TC,dice_ET
count,53.0000,53.0000,53.0000
mean,0.8320,0.7168,0.7706
std,0.1019,0.2913,0.2019
min,0.5702,0.0026,0.0158
25%,0.7571,0.6467,0.7471
50%,0.8787,0.8646,0.8512
75%,0.9117,0.9009,0.8850
max,0.9524,0.9557,0.9329


In [17]:
# Analise condicional 3/3 (B): Bimodalidade

df_bimodality_B = pd.DataFrame([
    {"region":         col,
     "pct_dice_zero": float((df_full_B[col] == 0.0).mean()) * 100,
     "pct_dice_one":  float((df_full_B[col] >= 0.999).mean()) * 100}
    for col, _ in pairs
])
df_bimodality_B.to_csv(LOGS / "cross_B_2024_on_2020_bimodality.csv", index=False)
df_bimodality_B.round(2)

,region,pct_dice_zero,pct_dice_one
0,dice_WT,0.0,0.0
1,dice_TC,0.0,0.0
2,dice_ET,0.0,0.0


In [18]:
# Metrica 3 (B): Lesion-wise Dice

lw_rows = []
for cid in tqdm(test_ids_2020, desc="Lesion-wise Dice por caso (B)"):
    gt = load_seg_2020(cid)
    pr = load_arr(PRED_2024_ON_2020 / f"{cid}.nii.gz").astype(np.int16)

    gt_wt = (gt==1)|(gt==2)|(gt==3)
    pr_wt = (pr==1)|(pr==2)|(pr==3)
    gt_tc = (gt==1)|(gt==3)
    pr_tc = (pr==1)|(pr==3)
    gt_et = (gt==3)
    pr_et = (pr==3)

    lw_rows.append({
        "id":     cid,
        "lwd_WT": lesion_wise_dice(gt_wt, pr_wt),
        "lwd_TC": lesion_wise_dice(gt_tc, pr_tc),
        "lwd_ET": lesion_wise_dice(gt_et, pr_et),
    })

lw_df_B = pd.DataFrame(lw_rows)
lw_df_B.to_csv(LOGS / "cross_B_2024_on_2020_lwd.csv", index=False)
lw_df_B[["lwd_WT","lwd_TC","lwd_ET"]].describe().round(4)

Lesion-wise Dice por caso (B): 100%|██████████| 53/53 [00:50<00:00,  1.05it/s]


,lwd_WT,lwd_TC,lwd_ET
count,53.0000,53.0000,53.0000
mean,0.5543,0.5518,0.5714
std,0.2987,0.3200,0.2856
min,0.0608,0.0004,0.0228
25%,0.2863,0.3052,0.2994
50%,0.4491,0.6129,0.5490
75%,0.8749,0.8677,0.8646
max,0.9524,0.9561,0.9330


In [19]:
# Tabela consolidada cross-dataset (medias) - apenas WT, TC, ET

summary = pd.DataFrame([
    {"direcao": "2020 -> 2024",
     "Dice WT": df_A["dice_WT"].mean(),  "Dice TC": df_A["dice_TC"].mean(),  "Dice ET": df_A["dice_ET"].mean(),
     "LWD WT":  lw_df_A["lwd_WT"].mean(), "LWD TC":  lw_df_A["lwd_TC"].mean(), "LWD ET":  lw_df_A["lwd_ET"].mean()},
    {"direcao": "2024 -> 2020",
     "Dice WT": df_B["dice_WT"].mean(),  "Dice TC": df_B["dice_TC"].mean(),  "Dice ET": df_B["dice_ET"].mean(),
     "LWD WT":  lw_df_B["lwd_WT"].mean(), "LWD TC":  lw_df_B["lwd_TC"].mean(), "LWD ET":  lw_df_B["lwd_ET"].mean()},
])
summary.to_csv(LOGS / "cross_summary_MoME.csv", index=False)
summary.round(4)

,direcao,Dice WT,Dice TC,Dice ET,LWD WT,LWD TC,LWD ET
0,2020 -> 2024,0.7411,0.3787,0.4922,0.5074,0.3144,0.4996
1,2024 -> 2020,0.8320,0.7168,0.7706,0.5543,0.5518,0.5714
